# Day 54: Hierarchical Clustering & DBSCAN

## Hierarchical Clustering
K-Means forces you to decide the number of clusters `k` upfront, which is often the hardest part of clustering. Agglomerative (bottom-up) hierarchical clustering avoids that: it starts with every point as its own cluster, then repeatedly merges the two closest clusters until only one remains, building a full tree ("dendrogram") of nested groupings. You cut the tree at whatever level gives you the number of clusters you want, so you can explore the structure before committing to `k`. This makes it especially useful for exploratory analysis — e.g. discovering natural groupings in customer or gene-expression data where you don't know how many segments actually exist. The code below generates three separated blobs of 2D points and uses `AgglomerativeClustering` with `n_clusters=3` to recover cluster labels for each point.

In [ ]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering

np.random.seed(0)
# Create 3 well-separated groups of 30 points each by shifting random Gaussian blobs
X = np.vstack([np.random.randn(30, 2), np.random.randn(30, 2) + 5, np.random.randn(30, 2) + [10, 0]])

# n_clusters=3 tells it where to cut the merge tree; default linkage ('ward')
# merges the pair of clusters that minimizes the increase in within-cluster variance
agg = AgglomerativeClustering(n_clusters=3)
labels = agg.fit_predict(X)
print(labels[:10])

## Dendrogram
A dendrogram is the visual record of every merge step in hierarchical clustering: the x-axis (or leaves) are individual points/small clusters, and the y-axis is the "distance" (dissimilarity) at which two clusters were merged. Reading it bottom-up, points that join together low down are very similar; the higher up a merge happens, the more dissimilar the two groups being combined are. A common way to pick a good number of clusters is to look for the tallest vertical gap the tree crosses without a merge — cutting there gives well-separated clusters. Here `linkage(X, method='ward')` computes the merge sequence (ward minimizes variance growth at each step, same criterion as the AgglomerativeClustering call above), and `truncate_mode='lastp', p=15` only shows the last 15 merges so the plot stays readable instead of showing every individual leaf.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt

# Compute the full merge sequence using Ward's method (minimizes variance increase per merge)
Z = linkage(X, method='ward')
# Only plot the last 15 merges (truncate_mode='lastp') so the tree is readable
dendrogram(Z, truncate_mode='lastp', p=15)
plt.title('Dendrogram')
plt.show()

## DBSCAN
DBSCAN (Density-Based Spatial Clustering of Applications with Noise) takes a fundamentally different approach from K-Means or hierarchical clustering: instead of grouping by distance to a center or by nested merges, it groups points that are densely packed together, and marks points sitting in low-density regions as noise/outliers rather than forcing them into a cluster. It needs two parameters instead of `k`: `eps`, the radius to search around each point, and `min_samples`, how many neighbors within that radius are needed for a point to count as a "core" point of a cluster. This makes DBSCAN good at finding clusters of arbitrary (non-spherical) shape and at flagging anomalies automatically — useful for things like detecting fraud clusters or sensor noise. In the output, cluster labels are 0, 1, 2, ... as usual, but label `-1` means "this point didn't fit into any dense region" (noise).

In [ ]:
from sklearn.cluster import DBSCAN

# eps=1.0: points within this radius of each other are considered neighbors
# min_samples=5: a point needs at least 5 neighbors within eps to be a "core" point
dbscan = DBSCAN(eps=1.0, min_samples=5)
labels_db = dbscan.fit_predict(X)
print('Unique labels (-1 is noise):', set(labels_db))

## When to Use Which
There's no single "best" clustering algorithm — the right choice depends on your data's shape and what you know in advance. K-Means is fast and scales well to large datasets, but you must choose `k` beforehand and it assumes clusters are roughly round (spherical) and similar in size, so it struggles with elongated or irregular groups. Hierarchical clustering needs no upfront `k` and gives you a dendrogram to explore different granularities, but it's slower (roughly O(n²) or worse) so it doesn't scale well to very large datasets. DBSCAN needs no `k` either, can find arbitrarily shaped clusters, and automatically flags outliers as noise, but it's sensitive to the `eps`/`min_samples` settings and struggles when clusters have very different densities. In practice: reach for K-Means when you have a rough idea of `k` and want speed, hierarchical when you want to explore structure at multiple scales, and DBSCAN when you expect noise/outliers or non-round clusters.

In [ ]:
# Quick recap of the trade-offs discussed above
print('K-Means -> fast, spherical clusters')
print('Hierarchical -> dendrogram, no k needed upfront')
print('DBSCAN -> arbitrary shapes, detects noise/outliers')